In [ ]:
import pandas as pd
import numpy as np

In [ ]:
bm_raw = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Consumption till 13th/bm_raw_new.csv")
bss_pca = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Consumption till 13th/bss_pca_raw.csv")
pca_sellers = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Consumption till 13th/PCA for Sellers.csv")
classification = pd.read_excel("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/3.Mar'26/display files/classification_mapping.xlsx")
Brand_mapping = pd.read_excel("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/3.Mar'26/display files/Brand_mapping.xlsx")


In [ ]:
bm_raw.columns

Index(['marketplace', 'campaign_id', 'advertiser_id', 'advertiser_name',
       'BUSINESS_ACCOUNT_ID', 'BUSINESS_ACCOUNT_NAME', 'seller_id',
       'adgroup_id', 'brand', 'commodity_name', 'analytic_vertical',
       'analytic_super_category', 'lifestyle_supercategory', 'Supercategory',
       'BU', 'HL_BU', 'Alpha_MP', 'BMP_Tag', 'Team', 'budget_type',
       'cost_model', 'pacing', 'campaign_type', 'campaign_name', 'status',
       'start_date', 'end_date', 'commodity_id', 'Channel', 'allocated_budget',
       'gross_budget', 'uniqueviewcount', 'actioncount', 'total_cost_x',
       'addtobasketcount', 'viewcount', 'cnt_lid', 'attr_units',
       'attr_revenue', 'overburn_share', 'Actual_spend', 'Budget_adgroup',
       'ROI', 'CTR', 'CVR'],
      dtype='object')

In [ ]:
bm_raw = bm_raw[bm_raw['total_cost_x']>5]

In [ ]:
# bss_pca.columns

In [ ]:
#  pca_sellers.columns

In [ ]:
classification.columns

Index(['Super Categories_Tag', 'BU_Tag', 'Wrong Tagging', 'New Tag', 'Type',
       'Adv. Name', 'Tag', 'Remarks', 'Super Categories_HL', 'BU_HL', 'BU_HL2',
       'HL Supercat'],
      dtype='object')

In [ ]:
Brand_mapping.columns

Index(['Incorrect Account Name', 'Correct Name', 'BU', 'Concatenate',
       'Advertiser Name', 'Tag', 'Ad Account ID', 'Ad Account Name',
       'Business Account ID', 'Business Account Name', 'Vertical',
       'Current SC', 'New SC'],
      dtype='object')

In [ ]:
incorrect_brands = Brand_mapping['Ad Account ID'].unique()

bm_raw['New Alpha/MP'] = np.where(
    bm_raw['advertiser_id'].isin(incorrect_brands) | (bm_raw['Team'] == 'IC'),
    'IC',
    bm_raw['Alpha_MP']
)


In [ ]:
bm_raw['New Alpha/MP'].value_counts()

,count
New Alpha/MP,
MP,452982
Alpha,42970
IC,2047


In [ ]:
bm_raw['FCFF Alpha/MP'] = np.where(bm_raw['marketplace'] == "HYPERLOCAL","HL",bm_raw['Alpha_MP'])

In [ ]:
bm_raw['FCFF Alpha/MP'].value_counts()

,count
FCFF Alpha/MP,
MP,454825
Alpha,36818
HL,6356


In [ ]:
brand_map = dict(zip(Brand_mapping['Vertical'], Brand_mapping['New SC']))
sc_map = dict(zip(classification['Wrong Tagging'], classification['New Tag']))

conditions = [
    bm_raw['marketplace'] == "Grocery",
    bm_raw['analytic_vertical'].isin(brand_map.keys()),
    bm_raw['analytic_super_category'].isin(brand_map.keys())
]

choices = [
    "Grocery",
    bm_raw['analytic_vertical'].map(brand_map),
    bm_raw['analytic_super_category'].map(brand_map)
]

fallback = bm_raw['analytic_super_category'].map(sc_map).fillna(bm_raw['analytic_super_category'])

bm_raw['New Supercat'] = np.select(conditions, choices, default=fallback)


In [ ]:
# print(bm_raw['New Supercat'].value_counts().to_string())


In [ ]:
bm_raw['SC match FCFF'] = bm_raw['New Supercat'].isin(classification['Super Categories_Tag'])

In [ ]:
# bm_raw['SC match FCFF'].value_counts()

In [ ]:
sc_bu_map = dict(zip(classification['Super Categories_Tag'], classification['BU_Tag']))

bm_raw['New BU'] = bm_raw['New Supercat'].map(sc_bu_map)

In [ ]:
print(bm_raw['New BU'].isnull().sum())

289


In [ ]:
bm_raw['Spends'] = bm_raw['total_cost_x']

In [ ]:
bm_raw['Spends'].sum()

np.float64(1159516729.84)

In [ ]:
brand_map = dict(zip(Brand_mapping['Incorrect Account Name'], Brand_mapping['Correct Name']))

bm_raw['lookup_key'] = bm_raw['advertiser_name'].astype(str) + bm_raw['analytic_super_category'].astype(str)

cond_a = bm_raw['lookup_key'].isin(Brand_mapping['Concatenate'])
cond_b = bm_raw['analytic_super_category'].isin(Brand_mapping['BU'])

bm_raw['Brand'] = np.where(
    cond_a & cond_b,
    bm_raw['advertiser_name'].map(brand_map),
    bm_raw['brand']
)
bm_raw['Brand'] = bm_raw['Brand'].fillna(bm_raw['brand'])


In [ ]:
len(bm_raw[bm_raw['Brand'] == "0"])


250

In [ ]:
# bm_raw['Brand'].value_counts(normalize=True)

# BSS PCA

In [ ]:
bss_pca.columns

Index(['Campaign ID', 'Marketplace', 'BU', 'Supercategory', 'HL_BU', 'store',
       'analytic_super_category', 'Brands', 'Store Name',
       'creative_template_id', 'creative_type', 'costmodel', 'Ad Account ID',
       'Ad Account Name', 'Business Account ID', 'Business Account Name',
       'Team', 'Alpha_MP', 'BMP_Tag', 'Channel', 'lifestyle_supercategory',
       'spend', 'views', 'clicks', 'ppv', 'click_direct_units',
       'click_indirect_units', 'click_direct_revenue',
       'click_indirect_revenue', 'Product', 'Average CPC', 'CTR', 'ROI',
       'New Alpha/MP', 'New Supercat', 'SC match FCFF', 'New BU', 'lookup_key',
       'Brand', 'FCFF Alpha/MP', 'BUxBrand', 'Check'],
      dtype='object')

In [ ]:
bss_pca = bss_pca[bss_pca['spend']>5].copy()

In [ ]:
incorrect_brands = Brand_mapping['Advertiser Name'].unique()

bss_pca['New Alpha/MP'] = np.where(
    bss_pca['Business Account Name'].isin(incorrect_brands) | (bss_pca['Team'] == 'IC'),
    'IC',
    bss_pca['Alpha_MP']
)


In [ ]:
bss_pca['New Alpha/MP'].value_counts()

,count
New Alpha/MP,
Alpha,23472
MP,8969


In [ ]:
class_map = dict(zip(classification['Wrong Tagging'], classification['New Tag']))

conditions = [ bss_pca['BU'] == "Grocery", bss_pca['BU'] == "Mobile", bss_pca['Supercategory'] == "PetCare", bss_pca['Supercategory'].isin(class_map.keys()) ]
choices = [ "Grocery", "Mobile", "Pets", bss_pca['Supercategory'].map(class_map) ]

bss_pca['New Supercat'] = np.select(conditions, choices, default=bss_pca['Supercategory'])


In [ ]:
bss_pca['New Supercat'].isnull().sum()

np.int64(0)

In [ ]:
bss_pca['SC match FCFF'] = bss_pca['New Supercat'].isin(classification['Super Categories_Tag'])

In [ ]:
bss_pca['SC match FCFF'].value_counts()

,count
SC match FCFF,
True,31973
False,468


In [ ]:
bss_pca[bss_pca['SC match FCFF'] == False]['New Supercat'].value_counts()

,count
New Supercat,
Not Assigned,468


In [ ]:
sc_bu_map = dict(zip(classification['Super Categories_Tag'], classification['BU_Tag']))

bss_pca['New BU'] = bss_pca['New Supercat'].map(sc_bu_map)

In [ ]:
print(bss_pca['New BU'].isnull().sum())

468


In [ ]:
brand_map = dict(zip(Brand_mapping['Incorrect Account Name'], Brand_mapping['Correct Name']))

bss_pca['lookup_key'] = bss_pca['Ad Account Name'].astype(str) + bss_pca['Supercategory'].astype(str)

cond_a = bss_pca['lookup_key'].isin(Brand_mapping['Concatenate'])
cond_b = bss_pca['Supercategory'].isin(Brand_mapping['BU'])

bss_pca['Brand'] = np.where(
    cond_a & cond_b,
    bss_pca['Ad Account Name'].map(brand_map),
    bss_pca['Brands']
)
bss_pca['Brand'] = bss_pca['Brand'].fillna(bss_pca['Brands'])

In [ ]:
len(bss_pca[bss_pca['Brand'] == "0"])

393

In [ ]:
bss_pca['FCFF Alpha/MP'] = np.where(bss_pca['Alpha_MP'] == "IC","Alpha",np.where(bss_pca['Alpha_MP'] =="3P","MP",bss_pca['Alpha_MP']))

In [ ]:
# bss_pca['FCFF Alpha/MP'].value_counts()

In [ ]:
bss_pca['BUxBrand'] = bss_pca['New BU'].astype(str) + bss_pca['Brands'].astype(str)
lookup_map = dict(zip(bss_pca['BUxBrand'], bss_pca['New Supercat']))

bss_pca['Check'] = bss_pca['BUxBrand'].map(lookup_map)

In [ ]:
# bss_pca.info()

In [ ]:

len(bss_pca[bss_pca['SC match FCFF'] == False])


468

In [ ]:
len(bss_pca[bss_pca['Supercategory'] == "Not Assigned"])

517

In [ ]:
bss_pca['New Supercat'] = np.where(bss_pca['Check'] == "Not Assigned", "Not Assigned",np.where(bss_pca['Check'] == "PetCare", "PetCare",bss_pca["Check"]))

In [ ]:
bss_pca['New BU'].value_counts()

,count
New BU,
BGM,14160
Lifestyle,4589
Electronics,4407
Grocery,3792
Home,3184
Large,1097
Furniture,449
Mobile,295


In [ ]:
# bss_pca['New Supercat'].value_counts()

In [ ]:
print(bss_pca['New BU'].isnull().sum())

468


In [247]:
bss_pca[bss_pca['SC match FCFF'] == False ]['spend'].sum()

np.float64(230599.01000000004)